In [1]:
import modules.data as d
import modules.model as m
import modules.train as t
import modules.utils as u

import torch
import torch.nn as nn
from pathlib import Path

# dataset dir
datasets = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')

# get device
device, generator = u.Devices().auto_set_device()

# get data
data = d.Data(
    # datasets
    tcga_project = 'TCGA-BRCA',
    metadata_subtype_col = 'paper_BRCA_Subtype_PAM50',

    # dirs
    tcga_dir = datasets / 'tcga',
    relation_filepath = datasets / 'other/relation_ohe.csv',
    
    # col, filter
    y_col = 'subtype',
    drop = {'subtype':['Normal', 'Metastatic']},
    max_subset=120,
)

*** Device() ***
device = cuda:2

**** Data() ****
log0_method      log1p            str
class_weights    (6,)             Tensor (cuda:2)
gene_counts      (4383, 567)      DataFrame
metadata         (567, 3)         DataFrame
relation         (32798, 18)      DataFrame
node_id_map      4383             dict
masks            305              list
X                (567, 4383, 1)   Tensor (cuda:2)
y                (567, 6)         Tensor (cuda:2)
y_labels         6                list
num_samples      567              int
num_nodes        4383             int
num_features     1                int
num_labels       6                int
num_masks        305              int



---

In [4]:
import pandas as pd
from typing import Literal

from pandas import DataFrame
from torch import Tensor
from modules.model import MLP

In [18]:
class MLPAutoencoder(nn.Module):
    def __init__(self, in_features:int, embedding_size:int=8, encoder_kwargs:dict={}, decoder_kwargs:dict={}, flatten:bool=False):
        super().__init__()

        # assign instance variables
        self.flatten = flatten
        
        # define layers
        self.encoder = MLP(
            in_features=in_features,
            out_features=embedding_size,
            **encoder_kwargs
        )
        self.decoder = MLP(
            in_features=embedding_size,
            out_features=in_features,
            **decoder_kwargs
        )

    def forward(self, X:Tensor):
        # flatten if applicable
        if self.flatten == True:
            X = X.squeeze(-1)

        # encode
        z = self.encoder(X)

        # decode
        X_hat = self.decoder(z)

        # unsqueeze
        if self.flatten == True:
            X_hat = X_hat.unsqueeze(-1)
            
        return X_hat

In [22]:
mlp = MLPAutoencoder(
    in_features=data.num_nodes,
    embedding_size=8,
    flatten=True
)

mlp_out = mlp(data.X)

In [30]:
mlp_out.shape

torch.Size([567, 4383, 1])

In [31]:
nn.MSELoss()(mlp_out, data.X)

tensor(1.6128, device='cuda:2', grad_fn=<MseLossBackward0>)

In [28]:
nn.L1Loss()(mlp_out, data.X)

tensor(1.0223, device='cuda:2', grad_fn=<MeanBackward0>)

In [29]:
nn.HuberLoss()(mlp_out, data.X)

tensor(0.6056, device='cuda:2', grad_fn=<HuberLossBackward0>)